# V1DD Functional Stimulus Metrics

Regenerating the `allen_v1dd` stimulus-response metrics — drifting gratings, surround
suppression, natural images, natural movie, receptive fields — from the NWB-Zarr
functional asset.

The original pipeline read a private Isilon HDF5 tree through `OPhysClient`, which no
longer exists; its own README says the code "would not work with the format you have
access to". So this is the same conversion as
**`Functional Data Cell-Cell Correlations.ipynb`**: keep the analysis, replace the data
access.

### Where this notebook currently is

| Milestone | Status |
|---|---|
| **M1 — schema truth** | this notebook |
| **M2 — response engine** | this notebook |
| **M3 — natural movie** (the deterministic end-to-end check) | this notebook |
| **M4 — drifting gratings → surround suppression** | this notebook |
| **M5 — natural images / images 12** | this notebook |
| **M6 — receptive fields** | this notebook |
| **M7 — packaging into the seven published tables** | this notebook |

M1 and M2 exist to *earn confidence before computing anything*. The per-trial stimulus
table this whole port depends on had never been read off a real file — its schema was
reconstructed from the NWB writer script — so M1 describes what is actually there. M2
proves the response arithmetic against a synthetic trace, where the right answer is known.

Each milestone writes a small JSON to `{save_dir}/checks/`, which is committed, so the
results can be reviewed away from the capsule.

In [1]:
import os
import sys
import time
from os.path import join as pjoin

import numpy as np
import pandas as pd
from IPython.display import display

# Robustly locate utils regardless of the kernel's working directory.
for _candidate in [pjoin("..", "utils"), pjoin("code", "utils"), "utils"]:
    if os.path.isdir(_candidate):
        sys.path.append(os.path.abspath(_candidate))
        break
else:
    raise FileNotFoundError(f"could not locate 'utils'; cwd={os.getcwd()}")

import stimulus_metrics as sm
import trial_responses as tr
import v1dd_nwb as vn
from checkpoints import checkpoint
from paths import resolve_data_root, resolve_dataset_dir

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
print(f"numpy {np.__version__} | pandas {pd.__version__}")

numpy 2.5.1 | pandas 2.3.3


## Paths

Input location comes from `utils/paths.py`, so this notebook finds its data on CodeOcean,
off the workshop USB drive, or in a local checkout without editing anything. Outputs
follow the correlations notebook's `scratch`/`results` knob.

In [2]:
mat_version = 1196

data_root = resolve_data_root(f"v1dd_{mat_version}")
functional_dir = resolve_dataset_dir("409828_V1DD_Filtered", root=data_root)

# The published allen_v1dd metric tables, for validation. Optional: without them the
# metrics still compute, they just cannot be checked against the original.
published_dir = resolve_dataset_dir("data_frames", root=data_root, required=False)

# Where derived tables and verification artifacts go.
output_target = "scratch"   # "scratch" (default, ephemeral) | "results" (reproducible run)
save_dir = pjoin(f"/{output_target}", f"v1dd_{mat_version}_coreg_functional_metrics")
os.makedirs(save_dir, exist_ok=True)

# The two sessions with EM coregistration. The correlations notebook derives this list
# live from CAVE; hard-coding it here keeps M1 free of a network dependency, and the
# schema report prints each session's own (column, volume) so a mismatch is visible.
TARGET_SESSIONS = [(1, "3"), (1, "5")]

print(f"data_root      : {data_root}")
print(f"functional_dir : {functional_dir}")
print(f"published_dir  : {published_dir}")
print(f"save_dir       : {save_dir}")

data_root      : /data/
functional_dir : /data/409828_V1DD_Filtered
published_dir  : /data/data_frames
save_dir       : /scratch/v1dd_1196_coreg_functional_metrics


## Session discovery

Each session's `(column, volume)` is read from the ROI table *inside* the file, never
from the directory name. If the correlations notebook has already been run, its cached
`session_index.csv` is reused — indexing every session from scratch takes about five
minutes.

The asset is **mixed-format**: most sessions are NWB-Zarr directories, a few are plain
HDF5 `.nwb` files. `find_sessions()` returns one path per session across both, and
`open_session()` dispatches on the suffix, so a bare `glob("*/*.nwb.zarr")` would quietly
come up short.

In [3]:
from pathlib import Path

# One path per session across both storage formats, Zarr preferred where a session has
# both. A bare glob for *.nwb.zarr silently drops the HDF5 sessions.
session_paths = vn.find_sessions(functional_dir)
_fmt = pd.Series([vn.nwb_format(p) for p in session_paths]).value_counts()
print(f"{len(session_paths)} session(s) in the asset  ({_fmt.to_dict()})")

# Reuse the correlations notebook's cached index if it is available.
_cached = resolve_dataset_dir(
    f"v1dd_{mat_version}_coreg_functional_correlation", root=data_root, required=False
)
if _cached and os.path.isfile(pjoin(_cached, "session_index.csv")):
    session_index = pd.read_csv(pjoin(_cached, "session_index.csv"))
    session_index["volume"] = session_index["volume"].astype(str)
    print(f"reusing cached session index from {_cached}")
else:
    print("no cached index found; peeking at each session (~5 min) ...")
    session_index = pd.DataFrame([vn.peek_session(p) for p in session_paths])
    _bad = session_index[session_index.get("error").notna()] if "error" in session_index else []
    if len(_bad):
        print(f"!! {len(_bad)} session(s) could not be read:")
        display(_bad[["name", "format", "error"]])

session_index["target"] = [
    (int(c), str(v)) in TARGET_SESSIONS
    for c, v in zip(session_index["column"], session_index["volume"])
]
targets = session_index.loc[session_index["target"]].reset_index(drop=True)
display(session_index)
print()
print(f"{len(targets)} target session(s):")
for _, r in targets.iterrows():
    print(f"  column {r['column']}, volume {r['volume']}  ->  {r['name']}")
if len(targets) != len(TARGET_SESSIONS):
    print()
    print(f"!! expected {len(TARGET_SESSIONS)} target sessions, found {len(targets)}")

25 session(s) in the asset  ({'zarr': 23, 'hdf5': 2})
reusing cached session index from /data/v1dd_1196_coreg_functional_correlation


,path,name,column,volume,n_planes,session_id,coregistered,target
0,/data/409828_V1DD_Filtered/409828_2018-11-06_1...,409828_2018-11-06_14-02-59_filtered_2026-04-09...,2,1,6,774328450,False,False
1,/data/409828_V1DD_Filtered/409828_2018-11-20_1...,409828_2018-11-20_10-42-45_filtered_2026-04-09...,3,1,6,783110306,False,False
2,/data/409828_V1DD_Filtered/409828_2018-11-21_0...,409828_2018-11-21_09-22-23_filtered_2026-04-16...,4,1,6,783878040,False,False
3,/data/409828_V1DD_Filtered/409828_2018-11-21_1...,409828_2018-11-21_10-56-07_filtered_2026-04-09...,4,2,6,784057573,False,False
4,/data/409828_V1DD_Filtered/409828_2018-11-26_1...,409828_2018-11-26_11-16-25_filtered_2026-04-09...,5,1,6,785378984,False,False
5,/data/409828_V1DD_Filtered/409828_2018-11-27_1...,409828_2018-11-27_11-01-58_filtered_2026-04-09...,2,2,6,785941763,False,False
6,/data/409828_V1DD_Filtered/409828_2018-11-27_1...,409828_2018-11-27_12-29-05_filtered_2026-04-09...,2,3,6,786071018,False,False
7,/data/409828_V1DD_Filtered/409828_2018-11-28_1...,409828_2018-11-28_10-54-56_filtered_2026-04-16...,3,3,6,786879416,False,False
8,/data/409828_V1DD_Filtered/409828_2018-11-29_1...,409828_2018-11-29_13-42-04_filtered_2026-04-09...,5,2,6,788220278,False,False
9,/data/409828_V1DD_Filtered/409828_2018-12-03_1...,409828_2018-12-03_14-25-24_filtered_2026-04-09...,4,3,6,790009715,False,False



2 target session(s):
  column 1, volume 3  ->  409828_2018-12-13_15-10-05_filtered_2026-04-09_05-57-20
  column 1, volume 5  ->  409828_2018-12-14_14-47-35_filtered_2026-04-09_06-13-08


## M1 — schema truth

`schema_report()` describes a session without computing any metric. It **reports rather
than asserts**: a missing table or column is recorded as an error string, because the
point is to learn what the file contains, and a function that raises on the first
surprise tells you much less than one that describes the whole file.

The things this needs to settle:

* Does `intervals['stimulus_table']` have the twelve expected columns? This schema was
  read off the NWB *writer* script and has never been verified against a real file.
  Everything downstream depends on it.
* Do the per-family trial counts match the denominators visible in the published CSVs —
  drifting gratings 8, natural images 8, natural images 12 **40**, natural movie **9**?
* Are there exactly 12 grating directions? The original hard-codes `(dir ± 3) % 12` for
  orthogonal directions and `(dir + 6) % 12` for the null direction, so any other number
  silently computes the wrong metric.
* Is the locally-sparse-noise template a 2× upsample of the 8×14 grid the receptive-field
  code was written against? **This decides whether M6 is possible at all.**
* Is `stop_time - start_time` equal to the 2.0 s the original took from an NWB attribute?
  If not, the response window is a decision rather than a lookup.

In [ ]:
%%time
reports = {}
for _, row in targets.iterrows():
    key = f"col{row['column']}_vol{row['volume']}"
    print(f"--- {key}  ({row['name']})")
    nwb, io = vn.open_session(row["path"])
    try:
        reports[key] = vn.schema_report(nwb)
        reports[key]["session"] = {
            "name": row["name"], "session_id": str(row["session_id"]),
            "column": int(row["column"]), "volume": str(row["volume"]),
        }
    finally:
        io.close()

path = checkpoint(
    "schema_report", reports, save_dir,
    sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()],
)

In [ ]:
# Compact verdict table -- the questions M1 exists to answer.
EXPECTED_TRIALS = {"drifting_gratings_full": 8, "drifting_gratings_windowed": 8,
                   "natural_images": 8, "natural_images_12": 40, "natural_movie": 9}

rows = []
for key, rep in reports.items():
    st = rep.get("stimulus_table", {})
    ps = rep.get("per_stimulus", {})
    lsn = rep.get("lsn_template", {})
    dgf = ps.get("drifting_gratings_full", {})

    rows.append({"session": key, "question": "stimulus_table present",
                 "answer": "error" not in st, "detail": st.get("error", f"{st.get('n_rows')} rows")})
    rows.append({"session": key, "question": "all 12 expected columns",
                 "answer": st.get("missing_vs_expected") == [],
                 "detail": f"missing={st.get('missing_vs_expected')} extra={st.get('extra_vs_expected')}"})
    rows.append({"session": key, "question": "exactly 12 grating directions",
                 "answer": dgf.get("n_directions") == 12,
                 "detail": str(dgf.get("n_directions"))})
    for fam, want in EXPECTED_TRIALS.items():
        got = (ps.get(fam) or {}).get("n_trials_inferred")
        rows.append({"session": key, "question": f"n_trials[{fam}] == {want}",
                     "answer": got == want, "detail": str(got)})
    rows.append({"session": key, "question": "one spontaneous block",
                 "answer": rep.get("epochs", {}).get("n_spontaneous_blocks") == 1,
                 "detail": str(rep.get("epochs", {}).get("n_spontaneous_blocks"))})
    rows.append({"session": key, "question": "is_soma == pika_conf > 0.5",
                 "answer": all(d.get("is_soma_matches_conf_gt_0.5") is True
                               for d in rep.get("planes", {}).get("detail", {}).values()),
                 "detail": ""})
    rows.append({"session": key, "question": "LSN template reduces to 8x14 (M6 viable)",
                 "answer": lsn.get("rf_viable") is True,
                 "detail": f"{lsn.get('native_shape')} -> {lsn.get('final_shape')}; "
                           f"uniform={lsn.get('blocks_uniform')}; {lsn.get('error', '')}"})
    dur = dgf.get("sweep_duration_s", {})
    rows.append({"session": key, "question": "DG stop-start == 2.0 s",
                 "answer": abs((dur.get("median_stop_minus_start") or 0) - 2.0) < 0.01,
                 "detail": f"stop-start={dur.get('median_stop_minus_start')}, "
                           f"onset-to-onset={dur.get('median_onset_to_onset')}"})

verdict = pd.DataFrame(rows)
display(verdict)
n_bad = int((~verdict["answer"].astype(bool)).sum())
print(f"\n{len(verdict) - n_bad}/{len(verdict)} checks answered as expected")
if n_bad:
    print("\nUnexpected answers -- read these before continuing to M3:")
    display(verdict.loc[~verdict["answer"].astype(bool)])

In [ ]:
# The stimulus table itself, for eyeballing. This is the artifact of record for M1.
nwb, io = vn.open_session(targets.iloc[0]["path"])
try:
    stim_table = vn.load_stimulus_table(nwb)
    epochs = vn.epoch_table(nwb)
finally:
    io.close()

print(f"stimulus_table: {stim_table.shape}")
display(stim_table.head(8))
print("\nsweeps per stimulus:")
display(stim_table["stim_name"].value_counts().rename("n_sweeps").to_frame())
print("\nepochs:")
display(epochs)

## M2 — response engine

`trial_responses.py` is the arithmetic layer: given traces, timestamps and stimulus onsets,
what was each neuron's mean activity in a window? The original asked this with a Python
loop over every sweep and every bootstrap draw, which costs 40–50 minutes for these two
sessions. Replacing it with a **prefix sum over time** makes each window mean two array
lookups, and the port runs in about five.

The subtlety worth stating: response windows land on a *variable* number of imaging
frames, because stimulus onsets are not frame-aligned. That looks like it forces a loop.
It does not — the samples are never materialised, so `b - a` is just an integer vector.

These checks use a synthetic trace where the right answer is known, so they prove the
engine independently of the data. Two of them are load-bearing:

* **Label-closed windows.** The original selected with `xarray.sel(time=slice(...))`,
  which includes *both* endpoints. A natural `(t >= lo) & (t < hi)` drops one sample per
  trial and shifts every response — the kind of difference that survives into a metric and
  looks like an algorithm bug.
* **Two different window primitives.** Trials use the label-closed form above; the
  bootstrap null uses a *frame-indexed*, fixed-width slice of `round(w / dt)` samples. At
  dt ≈ 0.164 s a 2 s grating window gives 13 samples for a trial and 12 for a null draw.
  That asymmetry is in the original, and reproducing its numbers means reproducing it.

In [ ]:
checks = {}
_rng = np.random.default_rng(0)
_n, _dt = 2000, 0.16374
_ts = np.cumsum(_rng.normal(_dt, _dt * 0.002, _n)) + 12.3      # jittered, like a real clock
_traces = _rng.gamma(2.0, 0.5, size=(_n, 7))                   # events-like, non-negative
_starts = _rng.uniform(_ts[5], _ts[-30], size=200)

# 1. label-closed windows, against the definition
_bad = 0
for w0, w1 in [(0.0, 2.0), (-1.0, 0.0), (0.0, 3 * _dt)]:
    a, b = tr.window_bounds(_ts, _starts, w0, w1)
    for i, s in enumerate(_starts):
        want = np.flatnonzero((_ts >= s + w0) & (_ts <= s + w1))   # BOTH ends inclusive
        if not np.array_equal(want, np.arange(a[i], b[i])):
            _bad += 1
checks["window_bounds_label_closed"] = {"mismatches": int(_bad), "n_tested": 600}

a, b = tr.window_bounds(_ts, _starts, 0.0, 2.0)
checks["window_width_varies"] = {"widths": sorted(int(w) for w in np.unique(b - a))}

# 2. prefix-sum means == direct slicing
cs, counts = tr.prefix_sums(_traces)
got = tr.window_means(cs, counts, a, b)
want = np.stack([_traces[a[i]:b[i]].mean(axis=0) for i in range(len(_starts))])
checks["window_means_vs_direct"] = {"max_abs_diff": float(np.max(np.abs(got - want)))}

# 3. nan-aware means
_tn = _traces.copy()
_tn[_rng.random(_tn.shape) < 0.02] = np.nan
csn, cn = tr.prefix_sums(_tn)
gotn = tr.window_means(csn, cn, a, b)
wantn = np.stack([np.nanmean(_tn[a[i]:b[i]], axis=0) for i in range(len(_starts))])
checks["nan_aware_means"] = {"max_abs_diff": float(np.nanmax(np.abs(gotn - wantn))),
                             "counts_array_built": cn is not None}

# 4. bootstrap null: fixed-width frame windows, and reproducible
_null = tr.spontaneous_null(_traces, _ts, _ts[100], _ts[900], (0.0, 2.0),
                            n_boot=500, rng=np.random.default_rng(42))
_again = tr.spontaneous_null(_traces, _ts, _ts[100], _ts[900], (0.0, 2.0),
                             n_boot=500, rng=np.random.default_rng(42))
checks["spontaneous_null"] = {
    "shape": list(_null.shape),
    "reproducible_under_seed": bool(np.array_equal(_null, _again)),
    "null_window_samples": int(round(2.0 / float(np.median(np.diff(_ts))))),
    "trial_window_samples_median": int(np.median(b - a)),
}

# 5. trial scatter, NaN padding, chronological rank
_resp = np.arange(16, dtype=float).reshape(8, 2)
_cond = np.array([0, 1, 2, 0, 1, 2, 0, 2])
_ta = tr.trial_array(_resp, _cond, n_trials=3, n_conditions=3)
checks["trial_array"] = {
    "shape": list(_ta.shape),
    "chronological_within_condition": bool(np.array_equal(_ta[0, :, 0], _resp[[0, 3, 6], 0])),
    "short_condition_nan_padded": bool(np.isnan(_ta[1, 2, 0])),
}

# 6. lifetime sparseness anchors
checks["lifetime_sparseness"] = {
    "uniform_is_0": float(tr.lifetime_sparseness(np.ones((1, 20)))[0]),
    "one_hot_is_1": float(tr.lifetime_sparseness(np.eye(1, 20)) [0]),
}

# 7. frac_trials_above_null excludes NaN trials rather than scoring them
_nl = np.tile(np.linspace(0, 1, 1000), (2, 1))
_trials = np.array([[2.0, 2.0, 2.0, 2.0], [0.5, 2.0, np.nan, np.nan]])
_fr = tr.frac_trials_above_null(_trials, _nl)
checks["frac_trials_above_null"] = {"all_strong": float(_fr[0]), "mixed_with_nan": float(_fr[1])}

ok = (checks["window_bounds_label_closed"]["mismatches"] == 0
      and checks["window_means_vs_direct"]["max_abs_diff"] < 1e-9
      and checks["nan_aware_means"]["max_abs_diff"] < 1e-9
      and checks["spontaneous_null"]["reproducible_under_seed"]
      and checks["trial_array"]["chronological_within_condition"]
      and checks["trial_array"]["short_condition_nan_padded"]
      and abs(checks["lifetime_sparseness"]["uniform_is_0"]) < 1e-12
      and abs(checks["lifetime_sparseness"]["one_hot_is_1"] - 1.0) < 1e-12
      and abs(checks["frac_trials_above_null"]["all_strong"] - 1.0) < 1e-12
      and abs(checks["frac_trials_above_null"]["mixed_with_nan"] - 0.5) < 1e-12)
checks["all_passed"] = bool(ok)

for k, v in checks.items():
    print(f"  {k}: {v}")
print(f"\nengine checks: {'ALL PASSED' if ok else 'FAILED -- do not continue to M3'}")

checkpoint("engine_tests", checks, save_dir, seed=0)

## M3 — natural movie

The first metric family, and deliberately the one with no bootstrap in it.

`frac_responsive_trials` for natural movie is not a statistical test: it is the fraction
of movie repeats whose mean response at the neuron's preferred frame is strictly greater
than zero. So comparing it against the published table exercises the stimulus table, the
trial scatter, the NaN padding, the response window and the argmax **with zero
stochasticity** — a hard pass/fail on the entire extraction path before any randomness is
introduced. `pref_img`, `pref_response` and `lifetime_sparseness` are equally
deterministic. Only `z_score` involves the bootstrap.

Two things about this family that are the original's design rather than ours: each movie
frame counts as a trial, and the response window spans about three imaging frames
(~0.49 s) while movie frames are 1/30 s apart — so consecutive "trials" overlap heavily.
`lifetime_sparseness` over 3,600 x 9 such values is therefore not measuring what its name
suggests. We reproduce it because the goal is to match the published table.

In [ ]:
%%time
rng_seed = 0
nm_tables = []

for _, srow in targets.iterrows():
    nwb, io = vn.open_session(srow["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        trials, _ = vn.stimulus_trials(stim, "natural_movie")
        spont = vn.spontaneous_block(nwb)
        for plane_key in vn.list_planes(nwb):
            plane = vn.load_plane(nwb, plane_key, trace_types=("events",))
            nm_tables.append(sm.natural_movie_metrics(
                plane, trials, spont,
                rng=np.random.default_rng(rng_seed),
            ))
            print(f"  col{plane.column} vol{plane.volume} {plane_key}: "
                  f"{plane.n_rois} ROIs, {len(trials)} sweeps")
            del plane
    finally:
        io.close()

nm_new = pd.concat(nm_tables, ignore_index=True)
print()
print(f"natural_movie: {len(nm_new)} ROIs across {len(nm_tables)} planes")
display(nm_new.head())

In [ ]:
# Compare against the published table. Joined on (column, volume, plane, roi) --
# never on roi_unique_id, which omits the column and collides across the five columns.
NM_METRICS = ["frac_responsive_trials", "lifetime_sparseness", "pref_img",
              "pref_response", "z_score"]

if published_dir is None:
    print("no published tables attached; skipping validation")
    nm_report = {"skipped": "published_dir not found"}
else:
    nm_pub = sm.load_published(published_dir, "natural_movie")
    nm_report = sm.compare_to_published(
        sm.to_published_schema(nm_new, "natural_movie"), nm_pub,
        NM_METRICS, exact=["pref_img"],
    )
    print(f"joined {nm_report['n_joined']} of {nm_report['n_new']} regenerated ROIs "
          f"({nm_report['n_only_published']} published rows have no NWB counterpart)")
    rows = []
    for m, v in nm_report["metrics"].items():
        rows.append({"metric": m, "n": v.get("n_both_finite"),
                     "max_abs_diff": v.get("max_abs_diff"),
                     "median_abs_diff": v.get("median_abs_diff"),
                     "frac_within_1e-9": v.get("frac_within_tol"),
                     "frac_exact": v.get("frac_exact"),
                     "pearson_r": v.get("pearson_r")})
    display(pd.DataFrame(rows))

    # The gate: frac_responsive_trials has no bootstrap, so it must match essentially
    # exactly. Anything else means the extraction path is wrong.
    gate = nm_report["metrics"]["frac_responsive_trials"]
    ok = gate.get("max_abs_diff", np.inf) < 1e-9
    print()
    print(f"GATE  frac_responsive_trials max_abs_diff = {gate.get('max_abs_diff')}"
          f"  ->  {'PASS' if ok else 'FAIL'}")
    if not ok:
        print("  The extraction path disagrees with the original. Suspects, in order:")
        print("   - response window (3 x imaging frame period)")
        print("   - trial scatter / NaN padding")
        print("   - argmax over frames picking a different preferred frame")

In [ ]:
nm_out = sm.to_published_schema(nm_new, "natural_movie")
nm_path = pjoin(save_dir, "natural_movie_M409828.csv")
nm_out.to_csv(nm_path, index=False)
print(f"wrote {nm_path}  ({len(nm_out)} rows)")

checkpoint("nm_validation", nm_report, save_dir, seed=rng_seed,
           sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])

# Per-ROI differences, for drilling into any metric that disagrees.
if published_dir is not None:
    merged = nm_out.merge(
        sm.load_published(published_dir, "natural_movie"),
        on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
    keep = ["column", "volume", "plane", "roi"]
    for m in NM_METRICS:
        merged[f"{m}_diff"] = (pd.to_numeric(merged[f"{m}_new"], errors="coerce")
                               - pd.to_numeric(merged[f"{m}_pub"], errors="coerce"))
        keep += [f"{m}_new", f"{m}_pub", f"{m}_diff"]
    per_roi = pjoin(save_dir, "checks", "nm_per_roi.csv")
    merged[keep].to_csv(per_roi, index=False)
    print(f"wrote {per_roi}  ({len(merged)} rows)")

## M4 — drifting gratings, then surround suppression

The biggest family, and the first one where the bootstrap reaches a *published* column.

Natural movie's `frac_responsive_trials` was `mean(response > 0)` — no randomness. Here it
is the fraction of preferred-condition trials beating a 2,500-draw spontaneous null at
p < 0.05, and `is_responsive` thresholds that at 0.5. So ROIs sitting near the boundary
will flip between random seeds, and there is no way to tell a flip from a bug without
knowing how much the metric moves on its own.

Hence the **two-seed control**: everything is computed twice with different seeds, and the
seed-A-vs-seed-B agreement is reported beside the seed-A-vs-published agreement. Only a
metric that agrees with the published table *materially worse than it agrees with itself*
is evidence of a problem.

Three things to watch:

* **`preferred_dir` / `preferred_sf`** are deterministic — they should match near-exactly.
  They also drive surround suppression, so a disagreement here propagates.
* **`osi` / `dsi` / `gosi` / `pref_dir_mean` / `lifetime_sparseness`** are deterministic
  given the trial responses. Expect float round-off, as in M3.
* **`frac_responsive_trials` / `is_responsive`** are the stochastic pair. Read them against
  the seed control, not against zero.

### The response-window decision

M1 measured drifting-grating sweeps at `stop_time - start_time` = **1.985 s**, while the
original took **2.0 s** from an NWB attribute it no longer has access to. `MetricConfig`
defaults to 2.0 to reproduce the published numbers. The cell after the comparison reruns
one plane at 1.985 s so the size of that choice is visible rather than assumed.

In [ ]:
%%time
DG_METRICS = ["dsi", "frac_responsive_trials", "gosi", "is_responsive",
              "lifetime_sparseness", "osi", "preferred_dir", "preferred_sf",
              "pref_dir_mean"]
SEEDS = (0, 1)          # two seeds: the second is the noise-floor control

dg_tables = {s: {"full": [], "windowed": []} for s in SEEDS}
ssi_tables = {s: [] for s in SEEDS}

for _, srow in targets.iterrows():
    nwb, io = vn.open_session(srow["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        spont = vn.spontaneous_block(nwb)
        running = vn.load_running_speed(nwb)
        trials = {}
        for dg_type in ("full", "windowed"):
            trials[dg_type] = vn.stimulus_trials(
                stim, f"drifting_gratings_{dg_type}", vn.DG_PARAM_COLUMNS)

        for plane_key in vn.list_planes(nwb):
            plane = vn.load_plane(nwb, plane_key, trace_types=("events",))
            for seed in SEEDS:
                # Tuning-curve fits depend only on the trial means, so they are
                # deterministic; refitting them for the control seed would double the
                # dominant cost (~29k curve_fit calls) for no information. ssi_tuning_fit
                # is therefore computed on the first seed only.
                cfg = sm.MetricConfig(fit_tuning_curves=(seed == SEEDS[0]))
                res = {}
                for dg_type in ("full", "windowed"):
                    t, blank = trials[dg_type]
                    res[dg_type] = sm.drifting_gratings_metrics(
                        plane, t, blank, spont, running, dg_type=dg_type, config=cfg,
                        rng=np.random.default_rng(seed))
                    dg_tables[seed][dg_type].append(res[dg_type].metrics)
                ssi_tables[seed].append(sm.surround_suppression_metrics(
                    res["windowed"], res["full"], plane))
            print(f"  col{plane.column} vol{plane.volume} {plane_key}: {plane.n_rois} ROIs")
            del plane
    finally:
        io.close()

dg_new = {s: {k: pd.concat(v, ignore_index=True) for k, v in d.items()}
          for s, d in dg_tables.items()}
ssi_new = {s: pd.concat(v, ignore_index=True) for s, v in ssi_tables.items()}
print()
print(f"drifting gratings: {len(dg_new[SEEDS[0]]['full'])} ROIs per stimulus type")
display(dg_new[SEEDS[0]]["windowed"].head())

In [5]:
# Agreement with the published table, beside agreement between the two seeds.
def agreement_table(new_a, new_b, published, metrics, exact=()):
    """One row per metric: how well it matches the original, and how well it matches
    itself under a different random seed. The second column is the noise floor."""
    vs_pub = sm.compare_to_published(new_a, published, metrics, exact=exact)
    vs_seed = sm.compare_to_published(new_a, new_b, metrics, exact=exact)
    rows = []
    for m in metrics:
        p, s = vs_pub["metrics"][m], vs_seed["metrics"][m]
        rows.append({
            "metric": m,
            "n": p.get("n_both_finite"),
            "vs_published_median": p.get("median_abs_diff"),
            "vs_seed_median": s.get("median_abs_diff"),
            "vs_published_max": p.get("max_abs_diff"),
            "vs_seed_max": s.get("max_abs_diff"),
            "r_published": p.get("pearson_r"),
            "exact_published": p.get("frac_exact"),
        })
    return pd.DataFrame(rows), vs_pub, vs_seed

In [ ]:
dg_reports, ssi_report = {}, None
if published_dir is None:
    print("no published tables attached; skipping validation")
else:
    for dg_type in ("full", "windowed"):
        fam = f"drifting_gratings_{dg_type}"
        pub = sm.load_published(published_dir, fam)
        tbl, vs_pub, vs_seed = agreement_table(
            sm.to_published_schema(dg_new[SEEDS[0]][dg_type], fam),
            sm.to_published_schema(dg_new[SEEDS[1]][dg_type], fam),
            pub, DG_METRICS, exact=["preferred_dir", "preferred_sf", "is_responsive"])
        dg_reports[fam] = {"vs_published": vs_pub, "vs_seed": vs_seed}
        print(f"--- {fam}  (joined {vs_pub['n_joined']} ROIs)")
        display(tbl)

    pub_ssi = sm.load_published(published_dir, "surround_supression_index")
    tbl, vs_pub, vs_seed = agreement_table(
        sm.to_published_schema(ssi_new[SEEDS[0]], "surround_supression_index"),
        sm.to_published_schema(ssi_new[SEEDS[1]], "surround_supression_index"),
        pub_ssi, sm.SSI_COLUMNS)
    ssi_report = {"vs_published": vs_pub, "vs_seed": vs_seed}
    print(f"--- surround suppression  (joined {vs_pub['n_joined']} ROIs)")
    display(tbl)

In [ ]:
# is_responsive is a threshold on a stochastic quantity, so report the confusion matrix
# rather than a correlation -- a 2% disagreement means different cells, not a smaller number.
if published_dir is not None:
    for dg_type in ("full", "windowed"):
        fam = f"drifting_gratings_{dg_type}"
        merged = sm.to_published_schema(dg_new[SEEDS[0]][dg_type], fam).merge(
            sm.load_published(published_dir, fam),
            on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
        a = merged["is_responsive_new"].fillna(0) > 0.5
        b = merged["is_responsive_pub"].fillna(0) > 0.5
        print(f"{fam}: agree {np.mean(a == b):.3%}  "
              f"(both yes {int((a & b).sum())}, both no {int((~a & ~b).sum())}, "
              f"new-only {int((a & ~b).sum())}, pub-only {int((~a & b).sum())})")
        # the same confusion, but between our two seeds -- the noise floor
        a2 = sm.to_published_schema(dg_new[SEEDS[1]][dg_type], fam)["is_responsive"] > 0.5
        print(f"{'':>{len(fam)}}  seed-to-seed agreement {np.mean(a.to_numpy() == a2.to_numpy()):.3%}")

In [ ]:
# What the response-window choice is worth. M1 measured stop-start at 1.985 s; the
# original used 2.0 s from an NWB attribute. One plane, both windows, same seed.
_row = targets.iloc[0]
nwb, io = vn.open_session(_row["path"])
try:
    stim = vn.load_stimulus_table(nwb)
    spont = vn.spontaneous_block(nwb)
    running = vn.load_running_speed(nwb)
    t, blank = vn.stimulus_trials(stim, "drifting_gratings_windowed", vn.DG_PARAM_COLUMNS)
    plane = vn.load_plane(nwb, vn.list_planes(nwb)[0], trace_types=("events",))
    window_variants = {}
    for secs in (2.0, 1.985):
        cfg = sm.MetricConfig(dg_response_seconds=secs)
        window_variants[secs] = sm.drifting_gratings_metrics(
            plane, t, blank, spont, running, dg_type="windowed", config=cfg,
            rng=np.random.default_rng(SEEDS[0])).metrics
finally:
    io.close()

cmp_win = sm.compare_to_published(
    sm.to_published_schema(window_variants[2.0], "drifting_gratings_windowed"),
    sm.to_published_schema(window_variants[1.985], "drifting_gratings_windowed"),
    DG_METRICS, exact=["preferred_dir", "preferred_sf"])
rows = [{"metric": m, "median_abs_diff": v.get("median_abs_diff"),
         "max_abs_diff": v.get("max_abs_diff"), "frac_exact": v.get("frac_exact"),
         "pearson_r": v.get("pearson_r")}
        for m, v in cmp_win["metrics"].items()]
print(f"2.0 s vs 1.985 s, one plane ({len(plane.roi)} ROIs):")
display(pd.DataFrame(rows))
del plane

In [ ]:
for dg_type in ("full", "windowed"):
    fam = f"drifting_gratings_{dg_type}"
    out = sm.to_published_schema(dg_new[SEEDS[0]][dg_type], fam)
    out.to_csv(pjoin(save_dir, f"{fam}_M409828.csv"), index=False)
    print(f"wrote {fam}_M409828.csv  ({len(out)} rows)")

ssi_out = sm.to_published_schema(ssi_new[SEEDS[0]], "surround_supression_index")
ssi_out.to_csv(pjoin(save_dir, "surround_supression_index_M409828.csv"), index=False)
print(f"wrote surround_supression_index_M409828.csv  ({len(ssi_out)} rows)")

checkpoint("dg_validation",
           {"seeds": list(SEEDS), "window_seconds": 2.0,
            "window_comparison_2p0_vs_1p985": cmp_win, **dg_reports},
           save_dir, seed=SEEDS[0],
           sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])
if ssi_report is not None:
    checkpoint("ssi_validation", {"seeds": list(SEEDS), **ssi_report}, save_dir,
               seed=SEEDS[0],
               sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])

# per-ROI differences for drilling in
if published_dir is not None:
    for fam, new in (("drifting_gratings_windowed", dg_new[SEEDS[0]]["windowed"]),
                     ("surround_supression_index", ssi_new[SEEDS[0]])):
        cols = DG_METRICS if fam.startswith("drifting") else sm.SSI_COLUMNS
        merged = sm.to_published_schema(new, fam).merge(
            sm.load_published(published_dir, fam),
            on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
        keep = ["column", "volume", "plane", "roi"]
        for m in cols:
            merged[f"{m}_diff"] = (pd.to_numeric(merged[f"{m}_new"], errors="coerce")
                                   - pd.to_numeric(merged[f"{m}_pub"], errors="coerce"))
            keep += [f"{m}_new", f"{m}_pub", f"{m}_diff"]
        path = pjoin(save_dir, "checks", f"{fam}_per_roi.csv")
        merged[keep].to_csv(path, index=False)
        print(f"wrote {path}  ({len(merged)} rows)")

## M5 — natural images and natural images 12

Structurally the same as natural movie — group trials by condition, take condition means,
find each neuron's preferred one — with two differences that matter.

**Conditions are `image_index`, not `image_order`.** `image_order` is the raw presentation
slot; `image_index` is the image's identity in the 118-image catalog. `natural_images_12`
draws twelve images from that *same* namespace, so its `pref_img` values are a sparse
subset of 0–117 (M1 measured them as 2, 4, 5, 6, 9, 23, 27, 29, 32, 47, 62, 68) rather
than 0–11. Re-ranking to 0–11 would look tidier and be wrong.

**`frac_responsive_trials` is a statistical test here.** Natural movie's version was
`mean(response > 0)` with no randomness; this one is the fraction of preferred-image
trials beating a 10,000-draw spontaneous null at p < 0.05. So it carries bootstrap noise
and gets the same two-seed control as the drifting gratings.

`natural_images_12` is also the heaviest single call in the pipeline: its multi-trial
null averages `n_trials = 40` draws per bootstrap sample, so 10,000 × 40 = 400,000 window
means. That is what `spontaneous_null`'s memory blocking is for.

### The response window is a recovered parameter

The original read the natural-images window from an NWB `duration_sec` attribute that the
current files no longer carry. M1 measured `stop_time - start_time` at 0.300 s and
onset-to-onset at 0.317 s, but neither is necessarily what the original used.

So rather than guess, the next cell **measures it**: run one plane at several candidate
windows and see which best reproduces the published table. This is legitimate here — we
are recovering a parameter the original used but did not record, and the published table
is the only witness — but it is a fit to the published data, so it is reported explicitly
rather than quietly baked in.

Worth knowing what can and cannot be distinguished: a window only changes anything if it
changes how many imaging samples fall inside each trial. At dt ≈ 0.165 s, windows of
0.25 s and 0.30 s put a mean of 1.52 and 1.81 samples in a trial respectively — different,
but not dramatically. Candidates closer together than that will be indistinguishable.

In [ ]:
%%time
# Probe the response window on one plane, per family, against the published table.
NI_METRICS = ["frac_responsive_trials", "lifetime_sparseness", "pref_img",
              "pref_response", "z_score"]
NI_DETERMINISTIC = ["lifetime_sparseness", "pref_img", "pref_response"]
# Two probes, because the first two rounds narrowed the question.
#
# Round 1 (0.20-0.35 s) improved monotonically to the ceiling. Round 2 (0.30-0.90 s)
# found an INTERIOR minimum at 0.35 s and ruled out the frame-based hypothesis: 0.495 s
# (3 imaging frames, natural movie's window) and 0.66 s (4 frames, LSN's) are far worse.
# But 0.35 s only reaches 1.9e-3 where the other families reach 1e-10, so no *duration*
# is exactly right.
#
# That points at the response primitive rather than its length. `lifetime_sparseness` is
# scale-invariant -- multiplying every response by a constant leaves it unchanged -- so
# it cannot be reporting a per-trial rescaling. But a *time window* rescales each trial
# differently, because the number of samples inside it depends on where the onset falls
# between frames. A *fixed frame count* does not. So the two models are distinguishable
# exactly by the metric that is failing.
NI_CANDIDATE_WINDOWS = [0.31, 0.33, 0.35, 0.37, 0.39]      # fine scan around the minimum
NI_CANDIDATE_FRAMES = [1, 2, 3, 4]                          # fixed samples from onset

probe_rows = []
if published_dir is None:
    print("no published tables attached; skipping the window probe")
    NI_WINDOW, NI_FRAMES = 0.30, None
    ni_cfg = sm.MetricConfig(ni_response_seconds=NI_WINDOW)
else:
    _row = targets.iloc[0]
    nwb, io = vn.open_session(_row["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        spont = vn.spontaneous_block(nwb)
        plane = vn.load_plane(nwb, vn.list_planes(nwb)[0], trace_types=("events",))
        plane_dt = plane.dt
        for fam in ("natural_images", "natural_images_12"):
            t, _ = vn.stimulus_trials(stim, fam)
            pub = sm.load_published(published_dir, fam)
            variants = ([("seconds", w, sm.MetricConfig(ni_response_seconds=w))
                         for w in NI_CANDIDATE_WINDOWS]
                        + [("frames", k, sm.MetricConfig(ni_response_frames=k))
                           for k in NI_CANDIDATE_FRAMES])
            for kind, value, cfg in variants:
                secs = value if kind == "seconds" else value * plane.dt
                got = sm.natural_images_metrics(plane, t, spont, ns_type=fam, config=cfg,
                                                rng=np.random.default_rng(0))
                rep = sm.compare_to_published(sm.to_published_schema(got, fam), pub,
                                              NI_METRICS, exact=["pref_img"])
                # Score on lifetime_sparseness alone. It is deterministic, continuous,
                # and uses every trial response rather than just the preferred one, so it
                # moves smoothly with the window. Averaging it with pref_img was a
                # mistake in the first probe: pref_img's median_abs_diff is 0 whenever
                # more than half the ROIs agree, which flattens the very signal being
                # measured.
                med = rep["metrics"]["lifetime_sparseness"]["median_abs_diff"]
                probe_rows.append({
                    "family": fam, "model": kind, "value": value,
                    "approx_seconds": round(secs, 4),
                    "lifetime_med": med,
                    "pref_img_exact": rep["metrics"]["pref_img"]["frac_exact"],
                    "pref_response_med": rep["metrics"]["pref_response"]["median_abs_diff"],
                    "score": med,
                })
        del plane
    finally:
        io.close()

    probe = pd.DataFrame(probe_rows)
    display(probe.pivot(index=["model", "value"], columns="family",
                        values=["lifetime_med", "pref_img_exact"]))

    curve = probe.groupby(["model", "value"])["score"].mean().sort_values()
    print()
    print("lifetime_sparseness median |diff|, both models (lower is better):")
    for (kind, value), v in curve.items():
        bar = "#" * max(1, int(40 * v / curve.max()))
        label = f"{value:.2f} s" if kind == "seconds" else f"{int(value)} frames"
        print(f"  {kind:<8} {label:>10}  {v:.3e}  {bar}")

    best_kind, best_value = curve.index[0]
    if best_kind == "frames":
        NI_WINDOW, NI_FRAMES = None, int(best_value)
        ni_cfg = sm.MetricConfig(ni_response_frames=NI_FRAMES)
        print()
        print(f"best: FIXED {NI_FRAMES} samples from onset  ({curve.iloc[0]:.3e})")
    else:
        NI_WINDOW, NI_FRAMES = float(best_value), None
        ni_cfg = sm.MetricConfig(ni_response_seconds=NI_WINDOW)
        print()
        print(f"best: {NI_WINDOW} s time window  ({curve.iloc[0]:.3e})")

    _converged = curve.iloc[0] < 1e-8
    _best_frames = curve.xs("frames").min() if "frames" in curve.index.get_level_values(0) else np.inf
    _best_secs = curve.xs("seconds").min() if "seconds" in curve.index.get_level_values(0) else np.inf
    print(f"  best fixed-frame {_best_frames:.3e}   vs   best time-window {_best_secs:.3e}")
    if _best_frames < _best_secs / 100:
        print("  -> the fixed-frame model wins decisively; the original averaged a set")
        print("     number of samples from onset, not everything inside a duration")
    print(f"converged? {_converged}  (the other families reach ~1e-10; anything above")
    print("  ~1e-6 means a systematic parameter is still wrong, not bootstrap noise)")

In [ ]:
%%time
ni_tables = {s: {"natural_images": [], "natural_images_12": []} for s in SEEDS}
# ni_cfg comes from the probe above -- either a duration or a fixed frame count

for _, srow in targets.iterrows():
    nwb, io = vn.open_session(srow["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        spont = vn.spontaneous_block(nwb)
        trials = {fam: vn.stimulus_trials(stim, fam)[0]
                  for fam in ("natural_images", "natural_images_12")}
        for plane_key in vn.list_planes(nwb):
            plane = vn.load_plane(nwb, plane_key, trace_types=("events",))
            for seed in SEEDS:
                for fam in ("natural_images", "natural_images_12"):
                    ni_tables[seed][fam].append(sm.natural_images_metrics(
                        plane, trials[fam], spont, ns_type=fam, config=ni_cfg,
                        rng=np.random.default_rng(seed)))
            print(f"  col{plane.column} vol{plane.volume} {plane_key}: {plane.n_rois} ROIs")
            del plane
    finally:
        io.close()

ni_new = {s: {k: pd.concat(v, ignore_index=True) for k, v in d.items()}
          for s, d in ni_tables.items()}
print()
print(f"natural images: {len(ni_new[SEEDS[0]]['natural_images'])} ROIs per family")
display(ni_new[SEEDS[0]]["natural_images_12"].head())

In [ ]:
ni_reports = {}
if published_dir is None:
    print("no published tables attached; skipping validation")
else:
    for fam in ("natural_images", "natural_images_12"):
        pub = sm.load_published(published_dir, fam)
        tbl, vs_pub, vs_seed = agreement_table(
            sm.to_published_schema(ni_new[SEEDS[0]][fam], fam),
            sm.to_published_schema(ni_new[SEEDS[1]][fam], fam),
            pub, NI_METRICS, exact=["pref_img"])
        ni_reports[fam] = {"vs_published": vs_pub, "vs_seed": vs_seed}
        print(f"--- {fam}  (joined {vs_pub['n_joined']} ROIs)")
        display(tbl)

    # pref_img is the identity, not the position: natural_images_12 must stay in the
    # 0-117 namespace rather than collapsing to 0-11.
    ids12 = sorted(set(sm.to_published_schema(
        ni_new[SEEDS[0]]["natural_images_12"], "natural_images_12").pref_img) - {-1})
    print(f"natural_images_12 pref_img values: {ids12}")
    print("  (should be a sparse subset of 0-117, not 0-11)")

In [ ]:
for fam in ("natural_images", "natural_images_12"):
    out = sm.to_published_schema(ni_new[SEEDS[0]][fam], fam)
    out.to_csv(pjoin(save_dir, f"{fam}_M409828.csv"), index=False)
    print(f"wrote {fam}_M409828.csv  ({len(out)} rows)")

checkpoint("ni_validation",
           {"seeds": list(SEEDS), "window_seconds": NI_WINDOW,
            "candidate_windows": NI_CANDIDATE_WINDOWS,
            "window_probe": probe_rows, **ni_reports},
           save_dir, seed=SEEDS[0],
           sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])

if published_dir is not None:
    for fam in ("natural_images", "natural_images_12"):
        merged = sm.to_published_schema(ni_new[SEEDS[0]][fam], fam).merge(
            sm.load_published(published_dir, fam),
            on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
        keep = ["column", "volume", "plane", "roi"]
        for m in NI_METRICS:
            merged[f"{m}_diff"] = (pd.to_numeric(merged[f"{m}_new"], errors="coerce")
                                   - pd.to_numeric(merged[f"{m}_pub"], errors="coerce"))
            keep += [f"{m}_new", f"{m}_pub", f"{m}_diff"]
        path = pjoin(save_dir, "checks", f"{fam}_per_roi.csv")
        merged[keep].to_csv(path, index=False)
        print(f"wrote {path}  ({len(merged)} rows)")

## M6 — receptive fields

The last family, and the odd one out in three ways.

**It uses dF/F, not deconvolved events** — the only family that does — and it is the only
one with a *subtracted* baseline (the 1 s before onset). Everything else uses events with
no baseline at all.

**There is no trial array and no GLM.** The published README describes receptive fields
computed "in a GLM framework". There is no regression anywhere in
`locally_sparse_noise.py`. It builds a design matrix recording which pixels were bright
and which dark on each sweep, then uses it purely as a counting indicator: a pixel's value
is the fraction of its presentations that produced a response above the ROI's own 95th
percentile of bootstrapped spontaneous activity. Fractions below 0.25 are zeroed, so
"has a receptive field" reduces to "at least one pixel survived", and the centre is the
**unweighted** centroid of the surviving pixel indices — the fractions are not weights.

**The pixel codes are read from the template, not hard-coded.** M1 found this asset
encodes the stimulus as −1 / 0 / +1 where the original assumed 0 / 127 / 255. A literal
port does not fail loudly: `pixel_on = 255` matches nothing, so every ON field silently
vanishes, while `pixel_off = 0` *collides with gray*, so the OFF map stops meaning "dark
pixel" and starts meaning "background pixel" — computed over roughly sixteen times as
many presentations. The result looks like data.

### Two known deviations, both reproduced on purpose

`point_to_alt_azi` divides the centre-to-centre range by `n` rather than `n − 1`, so its
degree scale is compressed by `(n−1)/n`: 12.5 % in altitude (8 rows), 7.1 % in azimuth
(14 columns). That is why the published tables span ±28.481° and ±56.132° instead of the
true ±32.55° and ±60.45°. `rf_center_scale_bug=False` gives the correct mapping.

The original also has a second, unused function (`rf_centers_argmax`) that transposes
azimuth and altitude. It is not in any published column and is not ported.

In [6]:
%%time
SEEDS = (0, 1)          # two seeds: the second is the noise-floor control
RF_METRICS = ["has_rf_on", "has_rf_off", "has_rf_on_or_off",
              "azimuth_rf_on", "altitude_rf_on", "azimuth_rf_off", "altitude_rf_off"]
RF_BOOL = ["has_rf_on", "has_rf_off", "has_rf_on_or_off"]

rf_tables = {s: [] for s in SEEDS}
for _, srow in targets.iterrows():
    nwb, io = vn.open_session(srow["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        spont = vn.spontaneous_block(nwb)
        trials, _ = vn.stimulus_trials(stim, "locally_sparse_noise")
        lsn = vn.load_lsn_template(nwb)
        if lsn.get("error"):
            raise RuntimeError(f"LSN template unusable: {lsn['error']}")
        print(f"  template {lsn['native_shape']} -> {lsn['images'].shape[1:]}, "
              f"pixel codes on={lsn['pixel_on']} off={lsn['pixel_off']} "
              f"gray={lsn['pixel_gray']}")

        for plane_key in vn.list_planes(nwb):
            # dF/F here, not events -- the only family that does
            plane = vn.load_plane(nwb, plane_key, trace_types=("dff",))
            for seed in SEEDS:
                rf_tables[seed].append(sm.receptive_field_metrics(
                    plane, trials, spont, lsn, rng=np.random.default_rng(seed)))
            print(f"  col{plane.column} vol{plane.volume} {plane_key}: {plane.n_rois} ROIs")
            del plane
    finally:
        io.close()

rf_new = {s: pd.concat(v, ignore_index=True) for s, v in rf_tables.items()}
_r = rf_new[SEEDS[0]]
print()
print(f"receptive fields: {len(_r)} ROIs")
print(f"  ON only {int((_r.has_rf_on & ~_r.has_rf_off).sum())}, "
      f"OFF only {int((~_r.has_rf_on & _r.has_rf_off).sum())}, "
      f"both {int((_r.has_rf_on & _r.has_rf_off).sum())}, "
      f"none {int((~_r.has_rf_on_or_off).sum())}")
display(_r.head())

  template (8, 14) -> (8, 14), pixel codes on=1 off=-1 gray=0
  col1 vol3 plane-0: 409 ROIs
  col1 vol3 plane-1: 470 ROIs
  col1 vol3 plane-2: 483 ROIs
  col1 vol3 plane-3: 478 ROIs
  col1 vol3 plane-4: 438 ROIs
  col1 vol3 plane-5: 430 ROIs
  template (8, 14) -> (8, 14), pixel codes on=1 off=-1 gray=0
  col1 vol5 plane-0: 193 ROIs
  col1 vol5 plane-1: 228 ROIs
  col1 vol5 plane-2: 202 ROIs
  col1 vol5 plane-3: 131 ROIs
  col1 vol5 plane-4: 90 ROIs
  col1 vol5 plane-5: 121 ROIs

receptive fields: 3673 ROIs
  ON only 320, OFF only 289, both 562, none 2502


,roi_unique_id,roi_key,mouse,column,volume,plane,roi,has_rf_on,has_rf_off,has_rf_on_or_off,azimuth_rf_on,altitude_rf_on,azimuth_rf_off,altitude_rf_off
0,M409828_3_0_0,M409828_13_0_0,M409828,1,3,0,0,True,True,True,-21.589286,-0.81375,-23.316429,-8.95125
1,M409828_3_0_1,M409828_13_0_1,M409828,1,3,0,1,False,False,False,NaN,NaN,NaN,NaN
2,M409828_3_0_2,M409828_13_0_2,M409828,1,3,0,2,False,False,False,NaN,NaN,NaN,NaN
3,M409828_3_0_3,M409828_13_0_3,M409828,1,3,0,3,False,False,False,NaN,NaN,NaN,NaN
4,M409828_3_0_4,M409828_13_0_4,M409828,1,3,0,4,False,False,False,NaN,NaN,NaN,NaN


CPU times: user 36.9 s, sys: 3.6 s, total: 40.5 s
Wall time: 5min 38s


In [7]:
rf_report = None
if published_dir is None:
    print("no published tables attached; skipping validation")
else:
    pub = sm.load_published(published_dir, "rf_metrics")
    a = sm.to_published_schema(rf_new[SEEDS[0]], "rf_metrics")
    b = sm.to_published_schema(rf_new[SEEDS[1]], "rf_metrics")
    # booleans compare as 0/1; the centres are continuous
    for f in (a, b, pub):
        for c in RF_BOOL:
            f[c] = f[c].astype(float)
    tbl, vs_pub, vs_seed = agreement_table(a, b, pub, RF_METRICS, exact=RF_BOOL)
    rf_report = {"vs_published": vs_pub, "vs_seed": vs_seed}
    print(f"--- rf_metrics  (joined {vs_pub['n_joined']} ROIs)")
    display(tbl)

    # A boolean that is a threshold on a bootstrap needs a confusion matrix, not a
    # correlation -- and the seed control says how much of any disagreement is noise.
    merged = a.merge(pub, on=["column", "volume", "plane", "roi"], how="inner",
                     suffixes=("_new", "_pub"))
    for c in RF_BOOL:
        x = merged[f"{c}_new"] > 0.5
        y = merged[f"{c}_pub"] > 0.5
        seed_b = b.set_index(["column", "volume", "plane", "roi"]).loc[
            merged.set_index(["column", "volume", "plane", "roi"]).index, c] > 0.5
        print(f"  {c:<18} vs published {np.mean(x == y):.3%}  "
              f"vs seed B {np.mean(x.to_numpy() == seed_b.to_numpy()):.3%}   "
              f"(new {int(x.sum())}, published {int(y.sum())})")

--- rf_metrics  (joined 3673 ROIs)


,metric,n,vs_published_median,vs_seed_median,vs_published_max,vs_seed_max,r_published,exact_published
0,has_rf_on,3673,0.0,0.0,1.000000,1.000000,0.926347,0.972502
1,has_rf_off,3673,0.0,0.0,1.000000,1.000000,0.934877,0.976586
2,has_rf_on_or_off,3673,0.0,0.0,1.000000,1.000000,0.933086,0.970596
3,azimuth_rf_on,853,0.0,0.0,51.814286,51.814286,0.948580,NaN
4,altitude_rf_on,853,0.0,0.0,28.481250,40.687500,0.949719,NaN
5,azimuth_rf_off,818,0.0,0.0,38.860714,38.860714,0.974428,NaN
6,altitude_rf_off,818,0.0,0.0,29.837500,29.837500,0.962510,NaN


  has_rf_on          vs published 97.250%  vs seed B 97.740%   (new 882, published 925)
  has_rf_off         vs published 97.659%  vs seed B 97.822%   (new 851, published 871)
  has_rf_on_or_off   vs published 97.060%  vs seed B 97.550%   (new 1171, published 1207)


In [8]:
# The scale bug is the sharpest single check on the centre mapping: regress our centres
# on the published ones. A slope of 1 means we reproduced their degree scale; a slope of
# n/(n-1) -- 1.143 in altitude, 1.077 in azimuth -- would mean we corrected it by mistake.
if published_dir is not None:
    pub = sm.load_published(published_dir, "rf_metrics")
    m = sm.to_published_schema(rf_new[SEEDS[0]], "rf_metrics").merge(
        pub, on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
    rows = []
    for col, n_pix in (("altitude_rf_on", 8), ("altitude_rf_off", 8),
                       ("azimuth_rf_on", 14), ("azimuth_rf_off", 14)):
        x = pd.to_numeric(m[f"{col}_pub"], errors="coerce").to_numpy()
        y = pd.to_numeric(m[f"{col}_new"], errors="coerce").to_numpy()
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() > 2:
            slope = float(np.polyfit(x[ok], y[ok], 1)[0])
            rows.append({"column": col, "n": int(ok.sum()), "slope_new_on_pub": slope,
                         "corrected_would_be": n_pix / (n_pix - 1),
                         "max_abs_diff": float(np.nanmax(np.abs(y[ok] - x[ok])))})
    display(pd.DataFrame(rows))
    print("slope ~1.000 confirms we reproduced the published (n-1)/n degree scale")

,column,n,slope_new_on_pub,corrected_would_be,max_abs_diff
0,altitude_rf_on,853,0.962914,1.142857,28.481250
1,altitude_rf_off,818,0.979625,1.142857,29.837500
2,azimuth_rf_on,853,0.974287,1.076923,51.814286
3,azimuth_rf_off,818,0.988344,1.076923,38.860714


slope ~1.000 confirms we reproduced the published (n-1)/n degree scale


In [9]:
rf_out = sm.to_published_schema(rf_new[SEEDS[0]], "rf_metrics")
rf_out.to_csv(pjoin(save_dir, "rf_metrics_M409828.csv"), index=False)
print(f"wrote rf_metrics_M409828.csv  ({len(rf_out)} rows)")

checkpoint("rf_validation",
           {"seeds": list(SEEDS), "scale_bug": True,
            "counts": {"on_only": int((_r.has_rf_on & ~_r.has_rf_off).sum()),
                       "off_only": int((~_r.has_rf_on & _r.has_rf_off).sum()),
                       "both": int((_r.has_rf_on & _r.has_rf_off).sum()),
                       "none": int((~_r.has_rf_on_or_off).sum())},
            **({"rf_metrics": rf_report} if rf_report else {})},
           save_dir, seed=SEEDS[0],
           sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])

if published_dir is not None:
    merged = rf_out.merge(sm.load_published(published_dir, "rf_metrics"),
                          on=["column", "volume", "plane", "roi"], how="inner",
                          suffixes=("_new", "_pub"))
    keep = ["column", "volume", "plane", "roi"]
    for m_ in RF_METRICS:
        keep += [f"{m_}_new", f"{m_}_pub"]
    path = pjoin(save_dir, "checks", "rf_metrics_per_roi.csv")
    merged[keep].to_csv(path, index=False)
    print(f"wrote {path}  ({len(merged)} rows)")

wrote rf_metrics_M409828.csv  (3673 rows)
  wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/rf_validation.json  (6.3 KB)
wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/rf_metrics_per_roi.csv  (3673 rows)


## M7 — packaging

The six metric sections above already wrote their seven CSVs in the published schema, so
nothing here recomputes anything. What is missing is the two artifacts that make the
results *usable* and *auditable* rather than merely correct.

**The wide table** is what students actually merge against `coreg_df`. Seven separate
CSVs on the same key is an invitation to seven separate merges, and five of the seven
carry a column literally called `lifetime_sparseness` — so a naive concatenation either
collides or silently suffixes to `lifetime_sparseness_x`. Every family therefore gets a
prefix, except the two whose published column names already carry theirs (`ssi_*`, and
the receptive-field columns `has_rf_on` / `azimuth_rf_on` / …). Renaming those to
`ssi_ssi_avg` would be worse than the inconsistency.

**The provenance record** exists because the defaults in `MetricConfig` are not obvious
and several of them are *deliberately wrong* — `rf_center_scale_bug`, `pref_cond_fillna`,
the 0.33 s natural-images window that was recovered by probing rather than read from the
data. A table of numbers with no record of which flags produced them cannot be checked by
anyone later, including us. So it records the defaults once and then each family's
**delta** from them, which is the part worth reading.

It also carries the seed. Nothing in the original pipeline was seeded — the published
numbers are one unreproducible draw from the global legacy `RandomState` — which is the
entire reason the two-seed control exists in the sections above.

In [ ]:
ID_COLS = ["roi_unique_id", "roi_key", "mouse", "column", "volume", "plane", "roi"]
KEYS = ["column", "volume", "plane", "roi"]

# (published family name, the seed-A frame, column prefix)
FAMILIES = {
    "dgf":  ("drifting_gratings_full",      dg_new[SEEDS[0]]["full"],              "dgf_"),
    "dgw":  ("drifting_gratings_windowed",  dg_new[SEEDS[0]]["windowed"],          "dgw_"),
    "ssi":  ("surround_supression_index",   ssi_new[SEEDS[0]],                     ""),
    "ni":   ("natural_images",              ni_new[SEEDS[0]]["natural_images"],    "ni_"),
    "ni12": ("natural_images_12",           ni_new[SEEDS[0]]["natural_images_12"], "ni12_"),
    "nm":   ("natural_movie",               nm_new,                                "nm_"),
    "rf":   ("rf_metrics",                  rf_new[SEEDS[0]],                      ""),
}

def _keyed(df):
    out = df.copy()
    out["volume"] = out["volume"].astype(str)
    for k in ("column", "plane", "roi"):
        out[k] = out[k].astype(int)
    return out

wide = None
manifest = {}
for short, (fam, raw, prefix) in FAMILIES.items():
    pub = _keyed(sm.to_published_schema(raw, fam))
    metric_cols = [c for c in pub.columns if c not in ID_COLS]
    part = pub[KEYS + metric_cols].rename(columns={c: prefix + c for c in metric_cols})
    if wide is None:
        wide = _keyed(raw[ID_COLS])
    # validate="one_to_one" is the assertion that matters: it fails loudly if any family
    # has a duplicate (column, volume, plane, roi), which would quietly fan the table out.
    wide = wide.merge(part, on=KEYS, how="left", validate="one_to_one")
    manifest[fam] = {"rows": int(len(pub)), "prefix": prefix,
                     "columns": [prefix + c for c in metric_cols]}

# Every family must cover exactly the same ROIs -- they are all built from the same planes
# in the same order, so anything else means a family silently dropped rows.
_ref = set(map(tuple, _keyed(nm_new)[KEYS].to_numpy().tolist()))
for short, (fam, raw, _) in FAMILIES.items():
    got = set(map(tuple, _keyed(raw)[KEYS].to_numpy().tolist()))
    if got != _ref:
        raise AssertionError(f"{fam}: ROI key set differs from natural_movie "
                             f"({len(got - _ref)} extra, {len(_ref - got)} missing)")
if not wide.columns.is_unique:
    dupes = wide.columns[wide.columns.duplicated()].tolist()
    raise AssertionError(f"duplicate columns in the wide table: {dupes}")
if len(wide) != len(nm_new):
    raise AssertionError(f"merge changed the row count: {len(nm_new)} -> {len(wide)}")

print(f"wide table: {len(wide)} ROIs x {len(wide.columns)} columns "
      f"({len(ID_COLS)} identity + {len(wide.columns) - len(ID_COLS)} metrics)")
for short, (fam, _, prefix) in FAMILIES.items():
    print(f"  {short:<5} {fam:<28} {len(manifest[fam]['columns']):>2} cols  "
          f"prefix {prefix!r}")
display(wide.head())

In [ ]:
# The seven CSVs are the published-compatible artifact; the feather is the convenient one.
# Feather because that is what the rest of this asset already uses -- the correlations
# output and `cell_cell_correlations_by_stimulus_coregistered.feather` are both feather,
# so students merging the two are not asked to learn a second format.
wide_path = pjoin(save_dir, "stimulus_metrics_M409828.feather")
try:
    wide.to_feather(wide_path)
except (ImportError, ValueError) as exc:
    wide_path = pjoin(save_dir, "stimulus_metrics_M409828.csv")
    wide.to_csv(wide_path, index=False)
    print(f"!! feather unavailable ({type(exc).__name__}: {exc}); wrote CSV instead")
print(f"wrote {wide_path}")

# Round-trip check. A wide table that reads back differently is worse than no wide table,
# and feather does silently change some dtypes (notably nullable ints and object columns).
_back = (pd.read_feather(wide_path) if wide_path.endswith(".feather")
         else pd.read_csv(wide_path, dtype={"volume": str}))
_bad = []
for c in wide.columns:
    a, b = wide[c], _back[c]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        same = np.allclose(a.astype(float), b.astype(float), rtol=0, atol=0, equal_nan=True)
    else:
        same = a.astype(str).equals(b.astype(str))
    if not same:
        _bad.append(c)
print(f"round-trip: {len(wide.columns) - len(_bad)}/{len(wide.columns)} columns identical"
      + (f"  !! differ: {_bad}" if _bad else ""))

In [ ]:
import dataclasses
import json
import platform
from datetime import datetime, timezone

from checkpoints import git_sha, jsonable


def config_dict(cfg):
    """MetricConfig -> plain dict. `dataclasses.asdict` cannot copy the MappingProxyType."""
    d = {f.name: getattr(cfg, f.name) for f in dataclasses.fields(sm.MetricConfig)}
    d["trace_type"] = dict(d["trace_type"])
    return d


DEFAULTS = config_dict(sm.MetricConfig())

# What each family actually ran with. Recorded as a delta from the defaults: reprinting
# twenty identical fields seven times would bury the three that differ.
CONFIG_USED = {
    "drifting_gratings_full":     sm.MetricConfig(fit_tuning_curves=True),
    "drifting_gratings_windowed": sm.MetricConfig(fit_tuning_curves=True),
    "surround_supression_index":  sm.MetricConfig(fit_tuning_curves=True),
    "natural_images":             ni_cfg,
    "natural_images_12":          ni_cfg,
    "natural_movie":              sm.MetricConfig(),
    "rf_metrics":                 sm.MetricConfig(),
}
config_deltas = {fam: {k: v for k, v in config_dict(c).items() if DEFAULTS[k] != v}
                 for fam, c in CONFIG_USED.items()}


def headline(report):
    """The single worst-agreeing metric in a family, beside the same metric's seed control.

    One number per family, chosen adversarially. A family is only worth investigating when
    the published column is materially worse than the seed column -- the seed column is
    what this pipeline achieves against itself, so it is the floor, not zero.
    """
    if not report:
        return None
    pub = report.get("vs_published", report)
    seed = report.get("vs_seed")
    scored = [(m, d["frac_within_tol"]) for m, d in pub["metrics"].items()
              if d.get("frac_within_tol") is not None]
    if not scored:
        return {"n_joined": pub.get("n_joined")}
    worst, val = min(scored, key=lambda kv: kv[1])
    entry = {"n_joined": pub.get("n_joined"), "worst_metric": worst,
             "frac_within_tol_vs_published": val,
             "pearson_r_vs_published": pub["metrics"][worst].get("pearson_r")}
    if seed and worst in seed.get("metrics", {}):
        entry["frac_within_tol_vs_seed"] = seed["metrics"][worst].get("frac_within_tol")
        entry["pearson_r_vs_seed"] = seed["metrics"][worst].get("pearson_r")
    return entry


def _ver(name):
    try:
        return __import__(name).__version__
    except Exception:
        return None


REPORTS = {
    "natural_movie": nm_report,
    "drifting_gratings_full": dg_reports.get("drifting_gratings_full"),
    "drifting_gratings_windowed": dg_reports.get("drifting_gratings_windowed"),
    "surround_supression_index": ssi_report,
    "natural_images": ni_reports.get("natural_images"),
    "natural_images_12": ni_reports.get("natural_images_12"),
    "rf_metrics": rf_report,
}

provenance = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_sha": git_sha(),
    "mouse": "M409828",
    "materialization_version": mat_version,
    "seeds": {"metrics": list(SEEDS), "natural_movie": rng_seed,
              "note": "seed[0] produced the published outputs; seed[1] is the noise-floor "
                      "control only. Nothing in the original pipeline was seeded."},
    "inputs": {"functional_dir": str(functional_dir),
               "published_dir": str(published_dir) if published_dir else None},
    "sessions": [{"name": r["name"], "column": int(r["column"]), "volume": str(r["volume"]),
                  "format": r.get("format"), "n_planes": int(r["n_planes"])}
                 for _, r in targets.iterrows()],
    "n_rois": int(len(wide)),
    "config_defaults": DEFAULTS,
    "config_deltas_per_family": config_deltas,
    "outputs": {"wide_table": os.path.basename(wide_path), "per_family": manifest},
    "validation": {fam: headline(rep) for fam, rep in REPORTS.items()},
    "environment": {"python": platform.python_version(), "platform": platform.platform(),
                    "packages": {m: _ver(m) for m in
                                 ("numpy", "pandas", "scipy", "pynwb", "hdmf", "hdmf_zarr",
                                  "h5py", "zarr", "pyarrow")}},
}

prov_path = pjoin(save_dir, "stimulus_metrics_provenance.json")
with open(prov_path, "w", encoding="utf-8") as fh:
    json.dump(jsonable(provenance), fh, indent=2, sort_keys=True, allow_nan=False)
    fh.write("\n")
print(f"wrote {prov_path}")
print()
_deviating = {f: d for f, d in config_deltas.items() if d}
if _deviating:
    print("families running off the defaults:")
    for f, d in _deviating.items():
        print(f"  {f}: {d}")
else:
    # Expected: the defaults ARE the published-matching settings, which is the point of
    # defaulting them that way. The values themselves are in `config_defaults`.
    print("all seven families ran at MetricConfig defaults")
print()
print("bug-compatibility flags in force (True == reproduce the published numbers):")
for k in ("rf_center_scale_bug", "pref_cond_fillna"):
    print(f"  {k} = {DEFAULTS[k]}")
print(f"  ni_response_seconds = {DEFAULTS['ni_response_seconds']} (recovered by probe, "
      f"not read from the data)")

_rows = [dict(family=f, **v) for f, v in provenance["validation"].items() if v]
if _rows:
    print()
    print("worst-agreeing metric per family, beside its seed-to-seed floor:")
    display(pd.DataFrame(_rows).set_index("family"))
else:
    print()
    print("no published tables attached, so nothing was validated")

In [ ]:
# Final manifest: what a consumer of this directory will find.
print(f"{save_dir}")
for f in sorted(os.listdir(save_dir)):
    p = pjoin(save_dir, f)
    if os.path.isfile(p):
        print(f"  {f:<48} {os.path.getsize(p) / 1024:>9.1f} KB")
    else:
        n = len(os.listdir(p))
        print(f"  {f + '/':<48} {n:>6} files")

_expected = [f"{fam}_M409828.csv" for fam, _, _ in FAMILIES.values()] + \
            [os.path.basename(wide_path), "stimulus_metrics_provenance.json"]
_missing = [f for f in _expected if not os.path.isfile(pjoin(save_dir, f))]
print()
print("all expected outputs present" if not _missing else f"!! missing: {_missing}")

## Running this on every session

The whole notebook is written against `targets`, so widening the scope is one assignment
— but it is not free, and the cost is not where you would guess.

The response engine is prefix-sum based, so its Python-level work does not scale with
trials or bootstraps; **it scales with planes**, because each plane is a separate
`load_plane` and a separate pass over the traces. The cell below prints the actual
multiplier for this asset from what already ran, rather than guessing.

Three things to know before you do it:

* **Only two sessions are EM-coregistered.** Every other session's metrics are perfectly
  valid, they just have no synaptic partner to merge against. That is a property of the
  coregistration, not of this pipeline.
* **Memory stays flat.** One plane's traces are live at a time and deleted at the end of
  each iteration, so peak RSS is set by the largest single plane, not by the session
  count. This matters on the 8 GB capsule.
* **The 3p sessions use letter volumes** (`a`–`f`). Everything here already treats
  `volume` as a string, but a CSV round-trip will re-infer `int` for an all-numeric
  column, so re-read with `dtype={"volume": str}` — `sm.load_published` already does.

In [ ]:
# What the full asset would cost, extrapolated from what actually ran.
_planes_done = int(targets["n_planes"].sum())
_ok = session_index[session_index["error"].isna()] if "error" in session_index else session_index
_planes_all = int(_ok["n_planes"].sum())
_csv_kb = sum(os.path.getsize(pjoin(save_dir, f)) for f in os.listdir(save_dir)
              if f.endswith(".csv")) / 1024
print(f"processed {len(targets)} session(s), {_planes_done} planes")
print(f"asset total {len(_ok)} session(s), {_planes_all} planes"
      f"  ->  x{_planes_all / _planes_done:.1f}")
print(f"multiply the M3-M6 %%time totals by that factor for the wall-clock estimate")
print(f"output size would grow from {_csv_kb:.0f} KB to about "
      f"{_csv_kb * _planes_all / _planes_done:.0f} KB")

# ---------------------------------------------------------------------------
# To run everything, replace the target selection in the session-discovery cell
# with the line below and re-run M3 through M7. Nothing else changes.
#
# targets = session_index[session_index["error"].isna()].reset_index(drop=True) \
#     if "error" in session_index else session_index.reset_index(drop=True)
#
# Sessions that failed to open are dropped rather than skipped mid-loop, so a bad file
# shortens the run instead of aborting it -- `peek_session` already recorded why.
#
# The published comparison keeps working unchanged: `compare_to_published` joins on
# (column, volume, plane, roi) and reports both set differences, so a session missing
# from either side shows up as a count rather than as a silent drop.
# ---------------------------------------------------------------------------

## Where this leaves things

All seven published tables regenerate from the NWB asset, and each family was checked
against the original **beside its own seed-to-seed noise floor** — without that control
the stochastic columns are uninterpretable, and it is what distinguished a real defect in
natural images from bootstrap jitter in drifting gratings.

Two things are worth revisiting, both deliberate:

**The natural-images response window is 0.33 s by recovery, not by record.** The original
read it from an NWB attribute these files no longer carry, and the value was found by
probing against the published table. It works because 0.33 s is just under `2 * dt`, so it
catches exactly two samples on every trial; the margin is 8e-5 s. If the imaging rate ever
changes, set `ni_response_frames=2` instead, which expresses the same intent and cannot
drift.

**The response windows were matched, not chosen.** They were tuned for the slow calcium
transients the original worked with, and this pipeline runs on deconvolved events, which
are far sparser. A window that is right for one is unlikely to be right for the other.
Matching first was the only way to prove the port is faithful — but now that it is proven,
the windows are the obvious thing to tune, and `MetricConfig` exists so that doing so is
a one-line change with the old behaviour still reachable.